# Debate Card Cutter RAG (FAISS + Groq)

This notebook builds a **local FAISS retriever** over your preprocessed card pairs and uses a Groq-hosted LLM to generate **tagged output** with `<ul>`, `<h>`, and `<e>` tags.

It is designed around your preprocessing format:
- `input_full.json`: `{tag, cite, text_body}`
- `output_full.json`: `{tag, cite, text_body}` where `text_body` contains inline formatting tags

The model is asked to return **tagged text**, not raw character offsets. After generation, the notebook:
1. validates the text was preserved exactly,
2. converts tags into spans,
3. renders a `.docx` card.

In [ ]:
!pip -q install groq sentence-transformers faiss-cpu python-docx tqdm

In [ ]:
!pip -q install -U google-genai

In [ ]:
import os
import re
import json
import time
import math
import hashlib
from pathlib import Path
from typing import List, Dict, Tuple
from getpass import getpass

import numpy as np
import faiss
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from groq import Groq

from docx import Document
from docx.enum.text import WD_COLOR_INDEX
from google import genai

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Config

Set your paths first. Keep the API key out of the notebook output; use `getpass()`.

In [ ]:
# ==== PATHS ====
INPUT_JSON_PATH = "/content/drive/MyDrive/cards_output_processed/input_full.json"      # or your Drive path
OUTPUT_JSON_PATH = "/content/drive/MyDrive/cards_output_processed/output_full.json"    # or your Drive path

# ==== RETRIEVAL / MODEL ====
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
GROQ_MODEL = "llama-3.3-70b-versatile"
TOP_K = 4
RETRIEVAL_TEXT_CHARS = 2200   # truncate stored retrieval text to keep prompts manageable
MAX_EXAMPLE_OUTPUT_CHARS = 2600
MAX_NEW_CARD_CHARS = 12000

# ==== OUTPUT DIR ====
ARTIFACT_DIR = "/content/card_cutter_rag_artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

print("Artifacts will be written to:", ARTIFACT_DIR)

In [ ]:
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass("Enter Gemini API key: ")

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

GEMINI_MODEL = "gemini-3-flash-preview"

## Load aligned examples

Your preprocessing notebook guarantees input/output alignment on `tag` and `cite`, with formatted tags only on the output `text_body`.

In [ ]:
path = INPUT_JSON_PATH

with open(path, "r", encoding="utf-8") as f:
    for i in range(3):
        line = f.readline()
        print(f"Line {i+1}: {line[:300]}")

In [ ]:
with open(INPUT_JSON_PATH, "r", encoding="utf-8") as f:
    input_data = json.load(f)
with open(OUTPUT_JSON_PATH, "r", encoding="utf-8") as f:
    output_data = json.load(f)

assert len(input_data) == len(output_data), "Input/output length mismatch"

examples = []
for i, (inp, out) in enumerate(zip(input_data, output_data)):
    assert inp["tag"] == out["tag"], f"Tag mismatch at {i}"
    assert inp["cite"] == out["cite"], f"Cite mismatch at {i}"
    examples.append({
        "id": i,
        "tag": inp["tag"],
        "cite": inp["cite"],
        "plain_text": inp["text_body"],
        "tagged_text": out["text_body"],
    })

print(f"Loaded {len(examples):,} aligned examples")
print("Sample keys:", list(examples[0].keys()))

In [ ]:
TAG_RE = re.compile(r"</?(ul|h|e)>")

def strip_tags(text: str) -> str:
    return TAG_RE.sub("", text)

def normalize_ws(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

bad = 0
for ex in examples[: min(len(examples), 2000)]:
    if normalize_ws(strip_tags(ex["tagged_text"])) != normalize_ws(ex["plain_text"]):
        bad += 1

print("Checked first", min(len(examples), 2000), "examples")
print("Mismatches after stripping tags + normalizing ws:", bad)

## Helper functions

In [ ]:
from typing import Dict

def build_retrieval_text(ex: Dict) -> str:
    # Include cite because debate cards often cluster strongly by author/publication.
    body = ex["plain_text"][:RETRIEVAL_TEXT_CHARS]
    return (
        f"TAG: {ex['tag']}\n"
        f"CITE: {ex['cite']}\n"
        f"BODY:\n{body}"
    )

def compact_example_for_prompt(ex: Dict) -> str:
    plain = ex["plain_text"][:RETRIEVAL_TEXT_CHARS]
    tagged = ex["tagged_text"][:MAX_EXAMPLE_OUTPUT_CHARS]
    return (
        f"Example Tag: {ex['tag']}\n\n"
        f"Example Cite: {ex['cite']}\n\n"
        f"Example Input Body:\n"
        f"{plain}\n\n"
        f"Example Output Body:\n"
        f"{tagged}"
    )

def count_tags(tagged_text: str) -> Dict[str, int]:
    return {
        "ul": tagged_text.count("<ul>"),
        "h": tagged_text.count("<h>"),
        "e": tagged_text.count("<e>"),
    }

In [ ]:
clean_examples = []
dropped = 0

for ex in examples:
    if normalize_ws(strip_tags(ex["tagged_text"])) == normalize_ws(ex["plain_text"]):
        clean_examples.append(ex)
    else:
        dropped += 1

examples = clean_examples
print(f"Kept {len(examples):,} clean aligned examples")
print(f"Dropped {dropped:,} mismatched examples")

## Build FAISS index

For now this is a pure dense retriever. Later, if you move to Pinecone, this cell becomes the embedding + upsert stage.

In [ ]:
embedder = SentenceTransformer(EMBED_MODEL, device="cuda")

# Use a subset first for speed. Set BUILD_LIMIT = None for all examples later.
BUILD_LIMIT = None
work_examples = examples if BUILD_LIMIT is None else examples[:BUILD_LIMIT]

retrieval_corpus = [build_retrieval_text(ex) for ex in work_examples]

embeddings = embedder.encode(
    retrieval_corpus,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("Using examples:", len(work_examples))
print("Embedding shape:", embeddings.shape)
print("FAISS index size:", index.ntotal)

In [ ]:
def retrieve_examples(new_tag: str, new_cite: str, new_body: str, top_k: int = TOP_K) -> List[Dict]:
    query = (
        f"TAG: {new_tag}\n"
        f"CITE: {new_cite}\n"
        f"BODY:\n{new_body[:RETRIEVAL_TEXT_CHARS]}"
    )

    q = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, idxs = index.search(q, top_k * 8)  # over-retrieve a little for deduping

    retrieved = []
    seen_ids = set()

    for score, idx in zip(scores[0], idxs[0]):
        ex = work_examples[int(idx)]
        if ex["id"] in seen_ids:
            continue
        seen_ids.add(ex["id"])
        retrieved.append({**ex, "score": float(score)})
        if len(retrieved) >= top_k:
            break

    return retrieved

In [ ]:
import os, pickle

SAVE_DIR = "/content/drive/MyDrive/debate_card_rag_faiss"
os.makedirs(SAVE_DIR, exist_ok=True)

faiss.write_index(index, os.path.join(SAVE_DIR, "cards.index"))

with open(os.path.join(SAVE_DIR, "cards_metadata.pkl"), "wb") as f:
    pickle.dump(work_examples, f)

print("Saved index and metadata to", SAVE_DIR)

In [ ]:
test_ex = work_examples[123]

hits = retrieve_examples(
    new_tag=test_ex["tag"],
    new_cite=test_ex["cite"],
    new_body=test_ex["plain_text"],
    top_k=3
)

for i, hit in enumerate(hits, 1):
    print(f"\n=== Hit {i} ===")
    print("Score:", round(hit["score"], 4))
    print("Tag:", hit["tag"][:200])
    print("Cite:", hit["cite"][:200])
    print("Plain preview:", hit["plain_text"][:500])

## Prompt + generation

We ask the model to return a JSON object with only one field: `tagged_text`.

Key constraints:
- preserve the exact original wording,
- add only `<ul>`, `<h>`, and `<e>` tags,
- no other edits,
- keep nesting sensible: `e` inside `h`, `h` inside `ul` whenever applicable.

In [ ]:
SYSTEM_PROMPT = """You format competitive debate evidence cards.

Your task is to add inline formatting tags to the BODY ONLY.

Allowed tags:
- <ul>...</ul>  for underlined text
- <h>...</h>    for highlighted text
- <e>...</e>    for emphasized/bold text

CRITICAL FORMATTING RULE:
You must write the angle brackets explicitly.
Use literal tags exactly like <h>text</h>, not htext/h.
Use literal tags exactly like <e>text</e>, not etext/e.
Use literal tags exactly like <ul>text</ul>, not ultext/ul.

Hard structural rules:
1. Every highlighted span must be fully inside an underlined span.
2. Every emphasized span must be fully inside a highlighted span.
3. Therefore, emphasized text must also be inside an underlined span.
4. Never produce a <h> span that is not nested inside <ul>.
5. Never produce a <e> span that is not nested inside <h> and <ul>.

Content rules:
6. Preserve the original body text exactly. Do not paraphrase, delete, add, or reorder words.
7. tagged_text must be the original body text plus only the allowed tags.
8. Keep formatting selective and human-like.
9. Do not tag the cite. Only tag the body text.
10. Return only the tagged body text.
11. Never output explanations or markdown fences.

Nesting examples:
- Valid: <ul>important <h>key <e>word</e></h> claim</ul>
- Invalid: <h>highlight without underline</h>
- Invalid: <e>emphasis outside highlight</e>
- Invalid: hhighlight/h
"""

def build_user_prompt(new_tag: str, new_cite: str, new_body: str, retrieved: List[Dict]) -> str:
    example_block = "\n\n---\n\n".join(
        compact_example_for_prompt(ex) for ex in retrieved
    )

    return (
        "Here are human-formatted examples retrieved from a debate-card corpus.\n"
        "Use them to imitate formatting style.\n"
        "Important: every <h> span must be inside <ul>, every <e> span must be inside <h>, "
        "and you must use literal angle brackets like <h>text</h>.\n\n"
        f"{example_block}\n\n"
        "Now format this new card body.\n\n"
        f"New Tag: {new_tag}\n"
        f"New Cite: {new_cite}\n"
        "New Input Body:\n"
        f"{new_body[:MAX_NEW_CARD_CHARS]}"
    )

In [ ]:
import json
import re
import html

ALLOWED_TAG_PATTERN = re.compile(r"</?(?:ul|h|e)>")

def repair_broken_inline_tags(text: str) -> str:
    text = html.unescape(text)

    text = re.sub(
        r'(?<![A-Za-z0-9_])(ul|h|e)([^<>\n]{1,400}?)/\1(?![A-Za-z0-9_])',
        lambda m: f"<{m.group(1)}>{m.group(2)}</{m.group(1)}>",
        text,
    )
    text = re.sub(
        r'(?<![A-Za-z0-9_])(ul|h|e)\s+([^<>\n]{1,400}?)/\1(?![A-Za-z0-9_])',
        lambda m: f"<{m.group(1)}>{m.group(2)}</{m.group(1)}>",
        text,
    )
    return text

def clean_tagged_text(tagged_text: str) -> str:
    tagged_text = tagged_text.strip()
    tagged_text = html.unescape(tagged_text)
    tagged_text = re.sub(r"^```(?:json)?\s*|\s*```$", "", tagged_text, flags=re.DOTALL).strip()
    tagged_text = repair_broken_inline_tags(tagged_text)

    def repl(match):
        token = match.group(0)
        return token if ALLOWED_TAG_PATTERN.fullmatch(token) else ""

    tagged_text = re.sub(r"<[^>]*>", repl, tagged_text)
    return tagged_text

def extract_tagged_text_from_response(raw: str) -> str:
    raw = raw.strip()

    try:
        obj = json.loads(raw)
        if isinstance(obj, dict) and "tagged_text" in obj:
            return clean_tagged_text(obj["tagged_text"])
    except Exception:
        pass

    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw, flags=re.DOTALL).strip()

    try:
        obj = json.loads(cleaned)
        if isinstance(obj, dict) and "tagged_text" in obj:
            return clean_tagged_text(obj["tagged_text"])
    except Exception:
        pass

    return clean_tagged_text(cleaned)

def gemini_generate_tagged_text(
    new_tag: str,
    new_cite: str,
    new_body: str,
    retrieved,
    temperature: float = 0.2,
):
    user_prompt = build_user_prompt(new_tag, new_cite, new_body, retrieved)

    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=[
            SYSTEM_PROMPT,
            user_prompt,
        ],
    )

    raw = response.text if hasattr(response, "text") and response.text else str(response)
    tagged_text = extract_tagged_text_from_response(raw)
    return {"tagged_text": tagged_text}, raw

In [ ]:
#test 1

test_ex = work_examples[123]
retrieved = retrieve_examples(
    new_tag=test_ex["tag"],
    new_cite=test_ex["cite"],
    new_body=test_ex["plain_text"],
    top_k=3
)

parsed, raw = groq_generate_tagged_text(
    new_tag=test_ex["tag"],
    new_cite=test_ex["cite"],
    new_body=test_ex["plain_text"],
    retrieved=retrieved,
)

print("RAW:")
print(raw[:2000])

print("\nVALID:", "tagged_text" in parsed)
print("\nTAGGED PREVIEW:\n")
print(parsed["tagged_text"][:3000])

In [ ]:
#test 2

test_ex = examples[15000]   # outside the 10k retrieval pool

retrieved = retrieve_examples(
    new_tag=test_ex["tag"],
    new_cite=test_ex["cite"],
    new_body=test_ex["plain_text"],
    top_k=3
)

parsed, raw = groq_generate_tagged_text(
    new_tag=test_ex["tag"],
    new_cite=test_ex["cite"],
    new_body=test_ex["plain_text"],
    retrieved=retrieved,
)

print("RAW:")
print(raw[:2000])

print("\nVALID:", "tagged_text" in parsed)
print("\nTEXT PRESERVED:",
      normalize_ws(strip_tags(parsed["tagged_text"])) == normalize_ws(test_ex["plain_text"]))

print("\nTAGGED PREVIEW:\n")
print(parsed["tagged_text"][:3000])

for i, ex in enumerate(retrieved, 1):
    print(f"\n=== Retrieved Example {i} ===")
    print("Score:", round(ex["score"], 4))
    print("Tag:", ex["tag"][:200])
    print("Cite:", ex["cite"][:200])
    print("Tagged preview:", ex["tagged_text"][:700])

## Validation and repair helpers

We reject generations that change the text. If needed, we retry once with a stricter reminder.

In [ ]:
def canonicalize_for_compare(text: str) -> str:
    return (
        text.replace("\u2018", "'")
            .replace("\u2019", "'")
            .replace("\u201c", '"')
            .replace("\u201d", '"')
            .replace("\u2013", "-")
            .replace("\u2014", "-")
            .replace("\u00a0", " ")
    )

def first_diff(a: str, b: str, context: int = 80):
    n = min(len(a), len(b))
    for i in range(n):
        if a[i] != b[i]:
            return {
                "idx": i,
                "a_context": a[max(0, i-context): i+context],
                "b_context": b[max(0, i-context): i+context],
            }
    if len(a) != len(b):
        i = n
        return {
            "idx": i,
            "a_context": a[max(0, i-context): i+context],
            "b_context": b[max(0, i-context): i+context],
        }
    return None

def validate_tagged_output(original_body: str, tagged_text: str) -> Tuple[bool, str]:
    tagged_text = clean_tagged_text(tagged_text)
    stripped = strip_tags(tagged_text)

    illegal = re.sub(r"</?(ul|h|e)>", "", re.sub(r"[^<>]", "", tagged_text))
    if illegal.strip():
        return False, f"Found unexpected angle-bracket content: {illegal[:120]}"

    if stripped == original_body:
        return True, "ok_exact"

    # Allow only character-substitution differences that preserve length/positioning
    if (
        len(stripped) == len(original_body)
        and canonicalize_for_compare(stripped) == canonicalize_for_compare(original_body)
    ):
        return True, "ok_canonical_same_length"

    diff = first_diff(stripped, original_body)
    if diff is None:
        return False, "Unknown mismatch"
    return (
        False,
        f"Mismatch at idx {diff['idx']}\n"
        f"generated: {repr(diff['a_context'])}\n"
        f"original : {repr(diff['b_context'])}"
    )

def retry_with_stricter_prompt(
    new_tag: str,
    new_cite: str,
    new_body: str,
    retrieved,
):
    strict_system = (
        SYSTEM_PROMPT
        + "\n\nCRITICAL: Copy the body text character-for-character exactly."
        + "\nKeep every punctuation mark, apostrophe, quote mark, dash, accent, and space exactly as given."
        + "\nOnly insert <ul>, </ul>, <h>, </h>, <e>, </e>."
        + "\nDo not normalize punctuation."
        + "\nReturn ONLY the tagged body text."
    )

    user_prompt = build_user_prompt(new_tag, new_cite, new_body, retrieved)

    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=[
            strict_system,
            user_prompt,
        ],
    )

    raw = response.text if hasattr(response, "text") and response.text else str(response)
    tagged_text = extract_tagged_text_from_response(raw)
    return {"tagged_text": tagged_text}, raw

## Tagged text → spans

In [ ]:
def merge_intervals(intervals: List[List[int]]) -> List[List[int]]:
    if not intervals:
        return []
    intervals = sorted([[int(s), int(e)] for s, e in intervals if e > s], key=lambda x: (x[0], x[1]))
    merged = [intervals[0]]
    for s, e in intervals[1:]:
        if s <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])
    return merged

def subtract_intervals(base: List[int], covered: List[List[int]]) -> List[List[int]]:
    s, e = base
    if e <= s:
        return []
    covered = merge_intervals([[max(s, cs), min(e, ce)] for cs, ce in covered if ce > s and cs < e])
    if not covered:
        return [[s, e]]

    out = []
    cur = s
    for cs, ce in covered:
        if cur < cs:
            out.append([cur, cs])
        cur = max(cur, ce)
    if cur < e:
        out.append([cur, e])
    return out

def ensure_nested_spans(spans: List[List]) -> List[List]:
    """
    Enforce:
      e inside h
      h inside ul
      therefore e inside ul
    Strategy:
      - keep all existing spans
      - add missing parent spans when needed
      - merge overlaps within each label
    """
    ul = [[s, e] for s, e, t in spans if t == "ul" and e > s]
    h  = [[s, e] for s, e, t in spans if t == "h" and e > s]
    e_ = [[s, e] for s, e, t in spans if t == "e" and e > s]

    ul = merge_intervals(ul)
    h  = merge_intervals(h)
    e_ = merge_intervals(e_)

    # Every e must be inside h: add h where missing
    extra_h = []
    for es, ee in e_:
        missing = subtract_intervals([es, ee], h)
        extra_h.extend(missing)
    h = merge_intervals(h + extra_h)

    # Every h must be inside ul: add ul where missing
    extra_ul = []
    for hs, he in h:
        missing = subtract_intervals([hs, he], ul)
        extra_ul.extend(missing)
    ul = merge_intervals(ul + extra_ul)

    repaired = []
    repaired += [[s, e, "ul"] for s, e in ul]
    repaired += [[s, e, "h"]  for s, e in h]
    repaired += [[s, e, "e"]  for s, e in e_]
    repaired.sort(key=lambda x: (x[0], x[1], x[2]))
    return repaired

def tagged_text_to_spans(tagged_text: str) -> Tuple[str, List[List]]:
    spans = []
    plain_chars = []
    stack = []  # list of (tag, start_idx)
    i = 0

    while i < len(tagged_text):
        if tagged_text.startswith("<ul>", i):
            stack.append(("ul", len(plain_chars)))
            i += 4
            continue

        if tagged_text.startswith("</ul>", i):
            for j in range(len(stack) - 1, -1, -1):
                if stack[j][0] == "ul":
                    tag, start = stack.pop(j)
                    spans.append([start, len(plain_chars), tag])
                    break
            i += 5
            continue

        if tagged_text.startswith("<h>", i):
            stack.append(("h", len(plain_chars)))
            i += 3
            continue

        if tagged_text.startswith("</h>", i):
            for j in range(len(stack) - 1, -1, -1):
                if stack[j][0] == "h":
                    tag, start = stack.pop(j)
                    spans.append([start, len(plain_chars), tag])
                    break
            i += 4
            continue

        if tagged_text.startswith("<e>", i):
            stack.append(("e", len(plain_chars)))
            i += 3
            continue

        if tagged_text.startswith("</e>", i):
            for j in range(len(stack) - 1, -1, -1):
                if stack[j][0] == "e":
                    tag, start = stack.pop(j)
                    spans.append([start, len(plain_chars), tag])
                    break
            i += 4
            continue

        plain_chars.append(tagged_text[i])
        i += 1

    plain_text = "".join(plain_chars)
    spans = [s for s in spans if s[1] > s[0]]
    spans = ensure_nested_spans(spans)
    return plain_text, spans

## DOCX renderer

In [ ]:
def render_docx_from_spans(tag: str, cite: str, text_body: str, spans: List[List], output_path: str):
    starts = {}
    ends = {}
    for s, e, t in spans:
        starts.setdefault(s, []).append(t)
        ends.setdefault(e, []).append(t)

    doc = Document()

    p = doc.add_paragraph()
    run = p.add_run(tag)
    run.bold = True

    p = doc.add_paragraph()
    p.add_run(cite)

    p = doc.add_paragraph()
    active = set()
    chunk = []

    def flush_chunk():
        nonlocal chunk
        if not chunk:
            return
        run = p.add_run("".join(chunk))
        if "ul" in active:
            run.underline = True
        if "h" in active:
            run.font.highlight_color = WD_COLOR_INDEX.YELLOW
        if "e" in active:
            run.bold = True
        chunk = []

    for idx, ch in enumerate(text_body):
        if idx in ends:
            flush_chunk()
            for t in sorted(ends[idx], reverse=True):
                if t in active:
                    active.remove(t)
        if idx in starts:
            flush_chunk()
            for t in sorted(starts[idx]):
                active.add(t)
        chunk.append(ch)

    flush_chunk()

    doc.save(output_path)

## End-to-end function

In [ ]:
import difflib
import math

def build_generated_to_original_map(generated_plain: str, original_plain: str):
    """
    Returns a list mapping each char index in generated_plain to a char index in original_plain.
    Uses SequenceMatcher opcodes and proportional mapping for replace blocks.
    """
    sm = difflib.SequenceMatcher(a=generated_plain, b=original_plain)
    mapping = [0] * len(generated_plain)

    for tag, a0, a1, b0, b1 in sm.get_opcodes():
        if tag == "equal":
            for i in range(a0, a1):
                mapping[i] = b0 + (i - a0)

        elif tag == "replace":
            a_len = a1 - a0
            b_len = b1 - b0
            if a_len == 0:
                continue
            for i in range(a0, a1):
                rel = (i - a0) / max(a_len - 1, 1)
                mapped = b0 + int(round(rel * max(b_len - 1, 0)))
                mapped = max(b0, min(mapped, max(b0, b1 - 1)))
                mapping[i] = mapped

        elif tag == "delete":
            # chars exist in generated but not original
            anchor = min(b0, max(len(original_plain) - 1, 0))
            for i in range(a0, a1):
                mapping[i] = anchor

        elif tag == "insert":
            # chars exist in original but not generated: no generated indices here
            pass

    # make mapping monotonic nondecreasing
    for i in range(1, len(mapping)):
        if mapping[i] < mapping[i - 1]:
            mapping[i] = mapping[i - 1]

    return mapping


def project_spans_to_original(
    generated_plain: str,
    original_plain: str,
    spans: List[List]
) -> List[List]:
    """
    Convert spans measured on generated_plain into spans on original_plain.
    """
    if generated_plain == original_plain:
        return spans

    if not generated_plain:
        return []

    char_map = build_generated_to_original_map(generated_plain, original_plain)

    projected = []
    for s, e, t in spans:
        if e <= s:
            continue

        s = max(0, min(s, len(generated_plain) - 1))
        e = max(1, min(e, len(generated_plain)))

        orig_s = char_map[s]
        orig_e = char_map[e - 1] + 1

        orig_s = max(0, min(orig_s, len(original_plain)))
        orig_e = max(orig_s, min(orig_e, len(original_plain)))

        if orig_e > orig_s:
            projected.append([orig_s, orig_e, t])

    # merge / re-enforce hierarchy using your existing helper
    projected = ensure_nested_spans(projected)
    return projected


def format_card_with_rag(tag: str, cite: str, text_body: str, top_k: int = TOP_K, save_prefix: str = "demo"):
    retrieved = retrieve_examples(tag, cite, text_body, top_k=top_k)
    print("Retrieved examples:")
    for i, ex in enumerate(retrieved, 1):
        print(f"{i}. id={ex['id']} score={ex['score']:.4f} tag={ex['tag'][:70]!r}")

    parsed, raw = gemini_generate_tagged_text(tag, cite, text_body, retrieved)
    tagged_text = parsed["tagged_text"]
    ok, msg = validate_tagged_output(text_body, tagged_text)

    if not ok:
        print("First pass failed validation:", msg)
        parsed, raw = retry_with_stricter_prompt(tag, cite, text_body, retrieved)
        tagged_text = parsed["tagged_text"]
        ok, msg = validate_tagged_output(text_body, tagged_text)
        print("Retry validation:", msg)

    generated_plain, spans = tagged_text_to_spans(tagged_text)

    # If exact validation failed, salvage by projecting spans back onto original text
    if not ok:
        print("Using span projection salvage path.")
        spans = project_spans_to_original(generated_plain, text_body, spans)
        recovered_plain = text_body
        validation_status = f"salvaged_projection | {msg}"
    else:
        recovered_plain = generated_plain
        validation_status = msg

    ts = int(time.time())
    json_path = os.path.join(ARTIFACT_DIR, f"{save_prefix}_{ts}_generation.json")
    docx_path = os.path.join(ARTIFACT_DIR, f"{save_prefix}_{ts}.docx")

    # Always render using the ORIGINAL text body
    render_docx_from_spans(tag, cite, text_body, spans, docx_path)

    artifact = {
        "tag": tag,
        "cite": cite,
        "text_body": text_body,
        "retrieved": [
            {
                "id": ex["id"],
                "score": ex["score"],
                "tag": ex["tag"],
                "cite": ex["cite"],
            }
            for ex in retrieved
        ],
        "tagged_text": tagged_text,
        "generated_plain": generated_plain,
        "spans": spans,
        "tag_counts": count_tags(tagged_text),
        "groq_model": GROQ_MODEL,
        "validation_status": validation_status,
    }

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(artifact, f, ensure_ascii=False, indent=2)

    print("Saved JSON artifact:", json_path)
    print("Saved DOCX:", docx_path)
    print("Tag counts:", artifact["tag_counts"])
    print("Total spans:", len(spans))
    print("Validation status:", validation_status)
    return artifact, json_path, docx_path

## Quick retrieval sanity check

In [ ]:
# Test retrieval on an existing example.
probe = examples[123]
retrieved = retrieve_examples(probe["tag"], probe["cite"], probe["plain_text"], top_k=4)
for i, ex in enumerate(retrieved, 1):
    print(f"{i}. score={ex['score']:.4f} id={ex['id']} tag={ex['tag'][:80]}")

## Interactive demo

In [ ]:
tag = input("Enter tag: ").strip()
cite = input("Enter cite: ").strip()
print("Paste text body. Finish with a line containing only END.")
lines = []

while True:
    line = input()
    if line == "END":
        break
    lines.append(line)

text_body = "\n".join(lines).strip()

artifact, json_path, docx_path = format_card_with_rag(
    tag,
    cite,
    text_body,
    top_k=TOP_K,
    save_prefix="interactive"
)

print("Done.")
print("JSON path:", json_path)
print("DOCX path:", docx_path)

## Optional: inspect the generated tagged text

In [ ]:
print(artifact["tagged_text"][:5000])

## Optional: download the generated DOCX in Colab

In [ ]:
from google.colab import files
files.download(docx_path)